# Temporal Pattern Analysis

## Goal

This notebook analyzes temporal patterns and time-based trends in the logistics data. 
The goal is to examine how delivery performance, transit durations, and order metrics 
fluctuate across different time horizons, such as seasons, months, and days of the week.

The following questions are examined:

1. Are there specific months or seasons associated with higher delivery delays?
2. How do delivery times and transit durations change over time?
3. Are there specific days of the week with consistently worse delivery performance?
4. Are there distinct temporal trends or cycles in order volume and revenue?

To answer these questions, time series metrics are aggregated across various temporal 
features (e.g., month, weekday, year-month). The results are presented through 
appropriate sequential and trend visualizations.

## Data Basis

The analysis primarily relies on the consolidated dataset processed by the ETL pipeline, 
focusing on timestamp features and performance metrics derived from:

- `loads` - shipment dates, order volumes, and financial metrics
- `trips` - dispatch and arrival timestamps
- `delivery_events` - specific event timing for pickup and delivery

Data cleaning, feature engineering, and metric calculations (`delivery_duration_hours`, 
`delivery_delay_hours`, `is_delayed_delivery`) happen in `src/delivery_pipeline.py`. 
This notebook loads the pre-processed output, extracts additional temporal features 
(such as `month`, `day_of_week`, and `year_month`), and focuses strictly on longitudinal 
analysis and visualization.

*Note on charts: **Matplotlib** is used as the base for most charts. For temporal 
trends, seasonality profiles, and aggregated time distributions, **Seaborn** is used on 
top of it to handle line plots, heatmaps, and categorical time series with built-in 
statistical aggregation.*

## 1. Load Data (via Pipeline)

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from delivery_pipeline import run_pipeline, PROCESSED_DATA_PATH_TEMPORAL

clean_df = run_pipeline(output_path=PROCESSED_DATA_PATH_TEMPORAL, include_routes=False)
clean_df = clean_df.drop(columns=["route_id"])
print(f"Loads used for analysis: {len(clean_df)}")

sns.set_theme(style="darkgrid")


*Note on data cleaning: Loads where the actual delivery timestamp is before the actual 
pickup timestamp were already excluded in the pipeline (see 
`drop_inconsistent_timestamps()` in `delivery_pipeline.py`). The raw data itself is not 
modified; less than 1% of loads were affected.*

### Data Quality Note

During timestamp validation, an edge case was found: one delivery had a rounded 
`delivery_duration_hours` value of 0.0, even though the exact timestamp comparison 
showed a (minimal) negative difference between the delivery and pickup timestamps. A 
plain filter on `delivery_duration_hours >= 0` would have let this inconsistent record 
pass as valid.

The check is therefore consistently performed on the unrounded timestamps 
(`delivery_actual_datetime >= pickup_actual_datetime`), not on the already rounded, 
derived metric, see `drop_inconsistent_timestamps()` in `delivery_pipeline.py`.

## 2. Extract Time Features

*Note on time features:* 
-`pickup_actual_datetime` is the base for all time feature calculations.
- Seasons are based on meteorological seasons, so
    - Spring: March 1 - May 31
    - Summer: June 1 – August 31
    - Fall: September 1 – November 30
    - Winter: December 1 – February 28 (February 29 in leap years)

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
clean_df["pickup_day_of_week"] = clean_df["pickup_actual_datetime"].dt.day_name()
clean_df["pickup_day_of_week"] = pd.Categorical(
    clean_df["pickup_day_of_week"],
    categories=weekday_order,
    ordered=True
)

clean_df["pickup_month"] = clean_df["pickup_actual_datetime"].dt.month
clean_df["pickup_year_month"] = clean_df["pickup_actual_datetime"].dt.to_period("M")

season_mapping = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall"
}

clean_df["pickup_season"] = clean_df["pickup_month"].map(season_mapping)


## 3. Seasonal and Monthly Patterns
### 3.1 Are delays consistent across months?

In [ ]:
monthly_stats = clean_df.groupby("pickup_month").agg(
    delay_rate=("is_delayed_delivery", "mean"),
    average_delay=("delivery_delay_hours", "mean"),
    average_duration=("delivery_duration_hours", "mean")
).round(4)
print(monthly_stats)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

sns.barplot(x=monthly_stats.index, y=monthly_stats["delay_rate"], ax=axes[0])
axes[0].set_title("Delivery Delay Rate by Month")
axes[0].set_ylabel("Delay Rate")
axes[0].tick_params(labelbottom=True)

sns.barplot(x=monthly_stats.index, y=monthly_stats["average_delay"], ax=axes[1])
axes[1].set_title("Average Delay by Month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Average Delay (hours)")
axes[1].tick_params(labelbottom=True)

plt.tight_layout()
plt.show()

*The delay rate ranges narrowly between roughly 0.66 and 0.68 across all months, and the
 average delay stays within about 1.48–1.57 hours. This suggests delivery delays are not
  strongly tied to a particular calendar month, there is no single month that stands out
   as substantially better or worse than the others.*

### 3.2 Are delays consistent across years (month by month)?

In [ ]:
yearly_stats = clean_df.groupby("pickup_year_month").agg(
    delay_rate=("is_delayed_delivery", "mean"),
    average_delay=("delivery_delay_hours", "mean"),
    average_duration=("delivery_duration_hours", "mean")
).round(4)
print(yearly_stats)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
labels = yearly_stats.index.strftime("%m-%y")
positions = range(0, len(yearly_stats), 2)

sns.barplot(x=yearly_stats.index, y=yearly_stats["delay_rate"], ax=axes[0])
axes[0].set_title("Delivery Delay Rate by Month and Year")
axes[0].set_xticks(positions)
axes[0].set_xticklabels(labels[::2], rotation=90)
axes[0].set_ylabel("Delay Rate")
axes[0].margins(x=0.01)
axes[0].tick_params(labelbottom=True)

sns.barplot(x=yearly_stats.index, y=yearly_stats["average_delay"], ax=axes[1])
axes[1].set_title("Average Delay by Month and Year")
axes[1].set_xlabel("Month-Year")
axes[1].set_xticks(positions)
axes[1].set_xticklabels(labels[::2], rotation=90)
axes[1].set_ylabel("Average Delay (hours)")
axes[1].margins(x=0.01)
axes[1].tick_params(labelbottom=True)

plt.tight_layout()
plt.show()

*Looking at the same metrics month-by-month across multiple years (2022–2024) shows a 
similar picture: values fluctuate within a comparably narrow band over time, without a 
clear upward or downward trend. This indicates that overall delivery performance has 
remained relatively stable across the observed period, rather than improving or 
deteriorating year over year.*

### 3.3 Are delays consistent across seasons?

In [ ]:
season_stats = clean_df.groupby("pickup_season").agg(
    delay_rate=("is_delayed_delivery", "mean"),
    average_delay=("delivery_delay_hours", "mean"),
    average_duration=("delivery_duration_hours", "mean")
).round(4)
print(season_stats)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5, 5), sharex=True)

sns.barplot(x=season_stats.index, y=season_stats["delay_rate"], ax=axes[0])
axes[0].set_title("Delivery Delay Rate by Season")
axes[0].set_ylabel("Delay Rate")
axes[0].tick_params(labelbottom=True)

sns.barplot(x=season_stats.index, y=season_stats["average_delay"], ax=axes[1])
axes[1].set_title("Average Delay by Season")
axes[1].set_xlabel("Season")
axes[1].set_ylabel("Average Delay (hours)")
axes[1].tick_params(labelbottom=True)

plt.tight_layout()
plt.show()

*Aggregating by season confirms the same pattern at a coarser level: delay rate and 
average delay are nearly identical across Winter, Spring, Summer, and Fall. Combined 
with the month-level results above, this points to delivery delays in this dataset being
 largely independent of seasonality.*

*Across all three granularities (month, year-month, season), delay metrics remain 
remarkably stable. This is a meaningful finding in itself: it suggests that whatever 
drives delivery delays in this dataset, it is not primarily a seasonal or calendar-based
 effect, pointing toward other factors (e.g. regional, operational, or customer-
 specific) as more likely drivers, which later notebooks in this series explore.*

## 4. Delivery Duration Over Time
### 4.1 How has delivery duration developed over time?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
positions = range(0, len(yearly_stats), 2)
labels = yearly_stats.index.strftime("%m-%y")

sns.lineplot(x=yearly_stats.index.astype(str),
             y=yearly_stats["average_duration"],
             marker="o",
             color="blue",
             ax=ax
            )
ax.set_title("Average Delivery Duration Over Time by Month and Year")
ax.set_xlabel("Month-Year")
ax.set_ylabel("Average Duration (hours)")
ax.set_xticks(positions)
ax.set_xticklabels(labels[::2], rotation=90)
ax.margins(x=0.01)

plt.tight_layout()
plt.show()


*Average delivery duration fluctuates within a narrow band of roughly one hour 
throughout the observed period (approximately 26–27 hours), closely matching the overall
average of 26.68 hours established in the delivery performance analysis. As with delay 
rate and average delay in Section 3, there is no visible long-term upward or downward 
movement, transit times appear stable over time rather than trending in either direction.*

### 4.2 Is there a discernible trend or cyclical pattern?

In [ ]:
# computing a 3 month rolling average
yearly_stats["three_month_rolling_avg_duration" ] = yearly_stats.average_duration.rolling(3).mean()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
positions = range(0, len(yearly_stats), 2)
labels = yearly_stats.index.strftime("%m-%y")

sns.lineplot(x=yearly_stats.index.astype(str),
             y=yearly_stats["average_duration"],
             marker="o",
             color="blue",
             ax=ax,
             label="Avg Duration"
            )

sns.lineplot(x=yearly_stats.index.astype(str),
             y=yearly_stats["three_month_rolling_avg_duration"],
             marker="o",
             color="orange",
             ax=ax,
             label="3-Month Rolling Avg Duration")

ax.set_title("Average Delivery Duration Over Time by Month and Year")
ax.set_xlabel("Month-Year")
ax.set_ylabel("Average Duration (hours)")
ax.set_xticks(positions)
ax.set_xticklabels(labels[::2], rotation=90)
ax.margins(x=0.01)

plt.tight_layout()
plt.show()

*The 3-month rolling average confirms this impression: it smooths out the small 
month-to-month fluctuations and stays essentially flat across the observed years, with 
no sustained increase or decrease. Combined with the seasonal findings from Section 3, 
this further reinforces that delivery duration - like delay - is not primarily driven by
time-based factors. Whatever affects performance in this dataset likely lies elsewhere, 
which the following notebooks will investigate (e.g. regional or operational drivers).*

## 5. Weekday Patterns
### 5.1 Are there specific days of the week with consistently worse delivery performance?

In [ ]:
daily_stats = clean_df.groupby("pickup_day_of_week").agg(
    delay_rate=("is_delayed_delivery", "mean"),
    average_delay=("delivery_delay_hours", "mean"),
    average_duration=("delivery_duration_hours", "mean")
).round(4)
print(daily_stats)

In [ ]:
short_labels = [day[:3] for day in daily_stats.index]
positions = range(len(daily_stats))

fig, axes = plt.subplots(2, 1, figsize=(5, 5), sharex=True)

sns.barplot(x=daily_stats.index, y=daily_stats["delay_rate"], ax=axes[0])
axes[0].set_title("Delivery Delay Rate by Day")
axes[0].set_ylabel("Delay Rate")
axes[0].set_xticks(positions)
axes[0].set_xticklabels(short_labels)
axes[0].tick_params(labelbottom=True)

sns.barplot(x=daily_stats.index, y=daily_stats["average_delay"], ax=axes[1])
axes[1].set_title("Average Delay by Day")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("Average Delay (hours)")
axes[1].set_xticks(positions)
axes[1].set_xticklabels(short_labels)
axes[1].tick_params(labelbottom=True)

plt.tight_layout()
plt.show()

*Delay rate ranges narrowly from 0.666 (Thursday) to 0.675 (Tuesday), and average delay 
from 1.48 to 1.55 hours, a spread of well under one percentage point in delay rate and 
roughly four minutes in average delay. Notably, Friday shows no elevated delay rate 
(0.673), so a potential pre-weekend peak is not supported by the data. As with the 
seasonal and duration findings above, delivery performance appears essentially 
independent of the day of the week.*

### 5.2 Do weekday effects interact with seasonality?

In [ ]:
weekday_month_pivot = clean_df.pivot_table(
    index="pickup_day_of_week",
    columns="pickup_month",
    values="is_delayed_delivery",
    aggfunc="mean"
)

sns.heatmap(weekday_month_pivot, annot=True, fmt=".2f", cmap="coolwarm")
plt.show()

*Breaking delay rate down by weekday and month simultaneously reveals a wider range 
(roughly 0.62–0.71) than either dimension shows on its own. However, this variation does 
not follow any consistent pattern across the grid, no weekday stands out as 
systematically worse in specific months, and no month amplifies a particular weekday's 
delay rate. Given that each weekday-month cell covers a considerably smaller sample than 
the month- or weekday-level aggregates, this spread is more plausibly explained by 
sampling noise than by a genuine interaction effect. This reinforces the broader 
conclusion of Section 3–5: delivery delays in this dataset show no meaningful dependence 
on time-based factors, whether considered individually or in combination.*

## 6. Order Volume and Revenue Trends
### 6.1 Are there distinct temporal trends in order volume?

In [ ]:
order_stats = clean_df.groupby("pickup_year_month").agg(
    order_count=("load_id", "count")
)

print(order_stats)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
positions = range(0, len(order_stats), 2)
labels = order_stats.index.strftime("%m-%y")

sns.barplot(x=order_stats.index.astype(str),
             y=order_stats["order_count"],
             ax=ax
            )
ax.set_title("Order count Over Time by Month and Year")
ax.set_xlabel("Month-Year")
ax.set_ylabel("Number of Orders")
ax.set_xticks(positions)
ax.set_xticklabels(labels[::2], rotation=90)
ax.margins(x=0.01)

plt.tight_layout()
plt.show()



In [ ]:
# computing a 3 month rolling average
order_stats["three_month_rolling_avg_order_count" ] = order_stats.order_count.rolling(3).mean()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
positions = range(0, len(order_stats), 2)
labels = order_stats.index.strftime("%m-%y")

sns.lineplot(x=order_stats.index.astype(str),
             y=order_stats["order_count"],
             marker="o",
             color="blue",
             ax=ax,
             label="Order Count"
            )

sns.lineplot(x=order_stats.index.astype(str),
             y=order_stats["three_month_rolling_avg_order_count"],
             marker="o",
             color="orange",
             ax=ax,
             label="3-Month Rolling Order Count")

ax.set_title("Order Count Over Time by Month and Year")
ax.set_xlabel("Month-Year")
ax.set_ylabel("Number of Orders")
ax.set_xticks(positions)
ax.set_xticklabels(labels[::2], rotation=90)
ax.margins(x=0.01)

plt.tight_layout()
plt.show()

Order volume fluctuates between roughly 2,120 and 2,485 orders per month, a wider 
relative spread (~15%) than the delay and duration metrics above, but still without any 
sustained upward or downward trend across the three years. The lowest values 
consistently appear in February each year (2022, 2023, 2024), which is most plausibly 
explained by the shorter calendar month rather than a genuine seasonal dip in demand. 
Overall, order volume appears stable over time, with day-count effects being the main 
source of month-to-month variation.

### 6.2 Are there distinct temporal trends in revenue?

In [ ]:
revenue_stats = clean_df.groupby("pickup_year_month").agg(
    avg_revenue=("revenue", "mean")
).round(4)
print(revenue_stats)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
positions = range(0, len(revenue_stats), 2)
labels = revenue_stats.index.strftime("%m-%y")

sns.barplot(x=revenue_stats.index.astype(str),
             y=revenue_stats["avg_revenue"],
             ax=ax
            )
ax.set_title("Revenue Over Time by Month and Year")
ax.set_xlabel("Month-Year")
ax.set_ylabel("Average Revenue")
ax.set_xticks(positions)
ax.set_xticklabels(labels[::2], rotation=90)
ax.margins(x=0.01)

plt.tight_layout()
plt.show()



In [ ]:
# computing a 3 month rolling average
revenue_stats["three_month_rolling_avg_revenue" ] = revenue_stats.avg_revenue.rolling(3).mean()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
positions = range(0, len(revenue_stats), 2)
labels = revenue_stats.index.strftime("%m-%y")

sns.lineplot(x=revenue_stats.index.astype(str),
             y=revenue_stats["avg_revenue"],
             marker="o",
             color="blue",
             ax=ax,
             label="Avg Revenue"
            )

sns.lineplot(x=revenue_stats.index.astype(str),
             y=revenue_stats["three_month_rolling_avg_revenue"],
             marker="o",
             color="orange",
             ax=ax,
             label="3-Month Rolling Avg Revenue")

ax.set_title("Average Revenue Over Time by Month and Year")
ax.set_xlabel("Month-Year")
ax.set_ylabel("Average Revenue")
ax.set_xticks(positions)
ax.set_xticklabels(labels[::2], rotation=90)
ax.margins(x=0.01)

plt.tight_layout()
plt.show()

Average revenue per load stays within a narrow band of roughly 3,030–3,165 across the 
entire period (a spread of under 5%), with no discernible long-term trend. Combined with 
the volume finding above, this suggests that neither the number of shipments nor their 
average value has meaningfully shifted over the three observed years; the business's 
core financial and operational metrics have remained remarkably steady.

## 7. Conclusion

This notebook examined delivery performance, transit duration, weekday effects, and 
order/revenue metrics across three years of data (2022–2024). Across every temporal 
granularity - month, year-month, season, and weekday, individually and in combination - 
delay rate, average delay, and delivery duration all remained within narrow, stable 
ranges with no discernible long-term trend. Order volume and revenue showed similarly 
stable patterns, with the only notable fluctuation (lower order counts in February) 
attributable to calendar day-count rather than genuine seasonal demand shifts.

The consistent absence of time-based patterns across every question examined here is 
itself a meaningful finding: it indicates that delivery performance in this dataset is 
not primarily driven by when a shipment occurs, but more likely by other operational 
factors, such as region, equipment type, carrier, or customer, that the following 
notebooks (`03_fleet_equipment_analysis`, `04_regional_analysis`, `05_customer_analysis`) 
are positioned to explore.

It is worth noting that the dataset is a realistic simulation rather than real 
operational data (per the Kaggle dataset description); this may account for the 
unusually uniform absence of temporal patterns observed throughout this notebook, which 
would be less typical in genuine historical logistics data.